# Load Data and Query

This notebook adds the unstructured layer — document chunks with vector embeddings — to the knowledge graph you built in Labs 1 and 2. After loading, you will run basic queries to verify the data and test vector search.

**Learning Objectives:**
- Load pre-computed chunk embeddings into an existing knowledge graph
- Create a vector index for semantic search
- Link entities to chunks with FROM_CHUNK relationships
- Run Cypher queries that traverse both structured and unstructured layers

## The Two-Layer Graph

Your graph already has a **structured layer** from Lab 1:
```
(:Company)-[:OFFERS]->(:Product)
(:Company)-[:FACES_RISK]->(:RiskFactor)
(:Company)-[:COMPETES_WITH]->(:Company)
(:AssetManager)-[:OWNS]->(:Company)
```

This notebook adds the **unstructured layer** — SEC filing text split into retrievable chunks:
```
(:Company)-[:FILED]->(:Document)<-[:FROM_DOCUMENT]-(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
```

And **cross-links** that connect entities to the chunks that mention them:
```
(:Product)-[:FROM_CHUNK]->(:Chunk)
(:RiskFactor)-[:FROM_CHUNK]->(:Chunk)
```

These cross-links are what make GraphRAG powerful. A vector search finds relevant chunks, then graph traversal reaches the structured entities — giving the LLM both the raw text and the knowledge graph context.

In [ ]:
%pip install neo4j python-dotenv -q

In [ ]:
import csv
import json
import os
from pathlib import Path

from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j!')

## Verify Existing Graph

Confirm the structured layer from Lab 1 is loaded before adding chunks.

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (n)
        WITH labels(n)[0] AS label, count(n) AS count
        RETURN label, count ORDER BY label
    """)
    print('=== Existing Graph ===')
    for record in result:
        print(f'  {record["label"]}: {record["count"]}')

## Load Seed Data Files

Read the pre-computed chunks (with 1024-dimensional embeddings) and relationship files from `setup/seed-data/`.

In [ ]:
SEED_DIR = Path('../setup/seed-data')

# Load chunks with embeddings
with open(SEED_DIR / 'chunks.jsonl') as f:
    chunks = [json.loads(line) for line in f]

# Load relationship files
def load_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))

chunk_documents = load_csv(SEED_DIR / 'chunk_documents.csv')
chunk_sequence = load_csv(SEED_DIR / 'chunk_sequence.csv')
entity_chunks = load_csv(SEED_DIR / 'entity_chunks.csv')

print(f'Chunks: {len(chunks)} (with {len(chunks[0]["embedding"])}-dim embeddings)')
print(f'FROM_DOCUMENT links: {len(chunk_documents)}')
print(f'NEXT_CHUNK links: {len(chunk_sequence)}')
print(f'FROM_CHUNK entity links: {len(entity_chunks)}')

## Create Chunk Nodes

Create a Chunk node for each piece of filing text. Each chunk carries its pre-computed embedding vector, so there is no need to call an embedding model.

In [ ]:
driver.execute_query(
    """UNWIND $chunks AS chunk
       CREATE (c:Chunk {chunkId: chunk.chunkId,
                        index: chunk.index,
                        text: chunk.text,
                        embedding: chunk.embedding})""",
    chunks=chunks,
)
print(f'Created {len(chunks)} Chunk nodes with embeddings')

## Create Vector Index

Create a vector index on the `embedding` property of Chunk nodes. This index uses cosine similarity and enables fast approximate nearest-neighbor search.

In [ ]:
driver.execute_query("""
    CREATE VECTOR INDEX chunkEmbeddings IF NOT EXISTS
    FOR (c:Chunk) ON (c.embedding)
    OPTIONS {indexConfig: {
        `vector.dimensions`: 1024,
        `vector.similarity_function`: 'cosine'
    }}
""")

print('Vector index created!')

## Link Chunks to Documents

Create `FROM_DOCUMENT` relationships linking each Chunk to the Document it came from. The Document nodes already exist from Lab 1.

In [ ]:
driver.execute_query(
    """UNWIND $rels AS rel
       MATCH (c:Chunk {chunkId: rel.chunkId})
       MATCH (d:Document {documentId: rel.documentId})
       CREATE (c)-[:FROM_DOCUMENT]->(d)""",
    rels=chunk_documents,
)
print(f'Created {len(chunk_documents)} FROM_DOCUMENT relationships')

## Link Consecutive Chunks

Create `NEXT_CHUNK` relationships to preserve the original reading order of the filing text.

In [ ]:
driver.execute_query(
    """UNWIND $rels AS rel
       MATCH (curr:Chunk {chunkId: rel.chunkId})
       MATCH (next:Chunk {chunkId: rel.nextChunkId})
       CREATE (curr)-[:NEXT_CHUNK]->(next)""",
    rels=chunk_sequence,
)
print(f'Created {len(chunk_sequence)} NEXT_CHUNK relationships')

## Link Entities to Chunks

Create `FROM_CHUNK` relationships connecting Products, RiskFactors, and other entities to the Chunks that mention them. These cross-links enable graph traversal from vector search results to structured knowledge.

In [ ]:
# Group by entity type for targeted MATCH queries
entity_type_map = {
    'Product': 'productId',
    'RiskFactor': 'riskId',
    'FinancialMetric': 'metricId',
    'Company': 'companyId',
}

total = 0
for entity_type, id_prop in entity_type_map.items():
    rels = [r for r in entity_chunks if r['entityType'] == entity_type]
    if not rels:
        continue
    driver.execute_query(
        f"""UNWIND $rels AS rel
            MATCH (e:{entity_type} {{{id_prop}: rel.entityId}})
            MATCH (c:Chunk {{chunkId: rel.chunkId}})
            CREATE (e)-[:FROM_CHUNK]->(c)""",
        rels=rels,
    )
    print(f'  {entity_type}: {len(rels)} FROM_CHUNK relationships')
    total += len(rels)

print(f'\nCreated {total} FROM_CHUNK relationships total')

## Verify the Complete Graph

Confirm that both layers — structured and unstructured — are present and connected.

In [ ]:
with driver.session() as session:
    # Node counts
    result = session.run("""
        MATCH (n)
        WITH labels(n)[0] AS label, count(n) AS count
        RETURN label, count ORDER BY label
    """)
    print('=== Node Counts ===')
    for record in result:
        print(f'  {record["label"]}: {record["count"]}')

    # Relationship counts
    result = session.run("""
        MATCH ()-[r]->()
        WITH type(r) AS type, count(r) AS count
        RETURN type, count
        ORDER BY type
    """)
    print('\n=== Relationship Counts ===')
    for record in result:
        print(f'  {record["type"]}: {record["count"]}')

## Test Queries

Run a few queries to explore the connected graph.

### Which companies have filing chunks loaded?

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (c:Company)-[:FILED]->(d:Document)<-[:FROM_DOCUMENT]-(chunk:Chunk)
        WITH c, d, count(chunk) AS chunks
        RETURN c.name AS company, c.ticker AS ticker,
               d.accessionNumber AS filing, chunks
        ORDER BY chunks DESC
    """)
    for record in result:
        print(f'{record["company"]} ({record["ticker"]}): {record["chunks"]} chunks — filing {record["filing"]}')

### Which products are mentioned in filing chunks?

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (p:Product)-[:FROM_CHUNK]->(chunk:Chunk)
        WITH p, count(chunk) AS mentions
        RETURN p.name AS product, mentions
        ORDER BY mentions DESC
        LIMIT 10
    """)
    print('Top 10 products by chunk mentions:')
    for record in result:
        print(f'  {record["product"]}: {record["mentions"]} chunk(s)')

### Traverse from a chunk to its company and products

This is the traversal pattern that the VectorCypherRetriever will use in the next notebooks — starting from a matched chunk and reaching structured entities.

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (chunk:Chunk {chunkId: 'CH001'})-[:FROM_DOCUMENT]->(doc:Document)
        OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
        OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(chunk)
        RETURN chunk.text AS text,
               company.name AS company,
               doc.accessionNumber AS filing,
               collect(DISTINCT product.name) AS products
    """)
    for record in result:
        print(f'Company: {record["company"]}')
        print(f'Filing: {record["filing"]}')
        print(f'Products mentioned: {record["products"]}')
        print(f'\nChunk text (first 200 chars):')
        print(f'  {record["text"][:200]}...')

## Test Vector Search

Run a raw vector similarity query to verify the index is working. This uses the `db.index.vector.queryNodes` procedure directly — the next notebooks wrap this in higher-level retrievers.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
from lib.data_utils import get_embedder

embedder = get_embedder()

query = "What are the main risk factors for technology companies?"
query_embedding = embedder.embed_query(query)

with driver.session() as session:
    result = session.run("""
        CALL db.index.vector.queryNodes('chunkEmbeddings', 3, $embedding)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)<-[:FILED]-(company:Company)
        RETURN company.name AS company,
               score,
               left(node.text, 150) AS preview
    """, embedding=query_embedding)

    print(f'Query: "{query}"\n')
    for record in result:
        print(f'{record["company"]} (score: {record["score"]:.4f})')
        print(f'  {record["preview"]}...\n')

## Summary

You extended the knowledge graph with an unstructured layer:

1. **Chunk nodes** with pre-computed 1024-dimensional embeddings from SEC 10-K filings
2. **Vector index** (`chunkEmbeddings`) for cosine similarity search
3. **FROM_DOCUMENT** relationships linking chunks to their source documents
4. **NEXT_CHUNK** relationships preserving reading order
5. **FROM_CHUNK** relationships connecting Products, RiskFactors, and other entities to the chunks that mention them

The graph now supports the full GraphRAG pattern: vector search finds relevant chunks, then graph traversal reaches the structured entities. The next notebooks build retrieval pipelines on top of this foundation.

---

**Next:** [Vector Retriever](02_vector_retriever.ipynb)

In [ ]:
driver.close()
print('Connection closed.')